In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test-setup").master("local[*]").getOrCreate()
print("Version Spark :", spark.version)

rdd = spark.sparkContext.parallelize(["big data spark", "spark docker", "big data"])
print(rdd.flatMap(lambda l: l.split(" ")).map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b).collect())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/08 20:22:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version Spark : 3.5.0


[Stage 0:====================================>                      (5 + 3) / 8]

[('spark', 2), ('docker', 1), ('big', 2), ('data', 2)]


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("check-raw").master("local[*]").getOrCreate()

df = spark.read.option("header", True).csv("/home/work/data/raw/olist_orders_dataset.csv")
print("Nombre de lignes :", df.count())
df.printSchema()
df.show(5, truncate=False)

26/11/08 20:22:42 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Nombre de lignes : 99441
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+

In [3]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("explore").master("local[*]").getOrCreate()
base = "/home/work/data/raw/"

noms = ["customers", "geolocation", "order_items", "order_payments",
        "order_reviews", "orders", "products", "sellers"]
tables = {n: spark.read.option("header", True).csv(f"{base}olist_{n}_dataset.csv") for n in noms}
tables["translation"] = spark.read.option("header", True).csv(f"{base}product_category_name_translation.csv")

# Test A : lignes et colonnes par table
for n, df in tables.items():
    print(f"{n:15} lignes={df.count():>8}  colonnes={len(df.columns)}")

26/11/08 20:22:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


customers       lignes=   99441  colonnes=5
geolocation     lignes= 1000163  colonnes=5
order_items     lignes=  112650  colonnes=7
order_payments  lignes=  103886  colonnes=5
order_reviews   lignes=  104162  colonnes=7
orders          lignes=   99441  colonnes=8
products        lignes=   32951  colonnes=9
sellers         lignes=    3095  colonnes=4
translation     lignes=      71  colonnes=2


In [4]:
df = spark.read.option("header", True).csv("/home/work/data/raw/olist_customers_dataset.csv")

In [5]:
df.printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [6]:
from pyspark.sql.functions import countDistinct, count

df.agg(
    countDistinct("customer_id").alias("valeurs_distinctes"),
    count("*").alias("total_lignes")
).show()

+------------------+------------+
|valeurs_distinctes|total_lignes|
+------------------+------------+
|             99441|       99441|
+------------------+------------+



In [7]:
from pyspark.sql.functions import countDistinct, count

df.agg(
    countDistinct("customer_unique_id").alias("valeurs_distinctes"),
    count("*").alias("total_lignes")
).show()

+------------------+------------+
|valeurs_distinctes|total_lignes|
+------------------+------------+
|             96096|       99441|
+------------------+------------+



In [8]:
(tables["customers"]
   .groupBy("customer_unique_id").count()
   .filter("count > 1")
   .orderBy(F.desc("count"))
   .show(5))


+--------------------+-----+
|  customer_unique_id|count|
+--------------------+-----+
|8d50f5eadf50201cc...|   17|
|3e43e6105506432c9...|    9|
|ca77025e7201e3b30...|    7|
|1b6c7548a2a1f9037...|    7|
|6469f99c1f9dfae77...|    7|
+--------------------+-----+
only showing top 5 rows



In [9]:
df = spark.read.option("header", True).csv("/home/work/data/raw/olist_order_items_dataset.csv")
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



In [10]:
df.show(15)

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.90|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.90|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.00|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.90|        18.14|
|00048cc3ae777c65d...|            1|ef92

In [11]:
from pyspark.sql.functions import countDistinct, count

df.agg(
    countDistinct("order_id").alias("valeurs_distinctes"),
    count("*").alias("total_lignes")
).show()

+------------------+------------+
|valeurs_distinctes|total_lignes|
+------------------+------------+
|             98666|      112650|
+------------------+------------+



In [12]:
# Q2a : order_items
items_par_commande = tables["order_items"].groupBy("order_id").count()
print("Commandes distinctes dans order_items :", items_par_commande.count())
print("Commandes avec plus d'une ligne       :", items_par_commande.filter("count > 1").count())


Commandes distinctes dans order_items : 98666
Commandes avec plus d'une ligne       : 9803


In [13]:
items_par_commande.filter("count>3").show()

+--------------------+-----+
|            order_id|count|
+--------------------+-----+
|118045506e1c1dda0...|    5|
|125e0c61fdbfc589f...|    4|
|29f49e25d09d4c903...|    6|
|3182ecdf3da0331bf...|    4|
|33fb9fcb4416d2b74...|    4|
|055cf2a6f9343f99a...|    6|
|0c87b31e37ed8557e...|    5|
|1bc2c38cb1ae7c211...|    5|
|13359bb817972f749...|    4|
|1782119bc21f8f920...|    4|
|204e562a760b68ad7...|    4|
|2050ad559df86365b...|    4|
|2aed9e5e9721d155d...|    4|
|309f6784abccce4e6...|    5|
|2455cbeb73fd04b17...|    6|
|2e5cfd97555ff101c...|    5|
|261f725152296e3e8...|    4|
|11d963f9aaaf7e748...|    4|
|193a665b822ac18f6...|    5|
|2b4d04cfb81d2dbb1...|    6|
+--------------------+-----+
only showing top 20 rows



In [14]:
payment_par_commande = tables["order_payments"].groupBy("order_id").count()
#payment_par_commande.show()
print("commandes distincts dans order payments :", payment_par_commande.count())
print("Commande avec plus d'un paiement :", payment_par_commande.filter("count > 1").count())

commandes distincts dans order payments : 99440
Commande avec plus d'un paiement : 2961


In [15]:
# Toutes les colonnes sont en string à ce stade, on caste price pour pouvoir l'additionner
items = tables["order_items"].withColumn("price", F.col("price").cast("double"))
paiements = tables["order_payments"]

# CA correct : somme directe sur order_items
ca_correct = items.agg(F.sum("price")).first()[0]

# CA après jointure naïve avec les paiements
ca_gonfle = items.join(paiements, "order_id").agg(F.sum("price")).first()[0]

print(f"CA correct : {ca_correct:,.2f}")
print(f"CA gonflé  : {ca_gonfle:,.2f}")
print(f"Écart      : {(ca_gonfle / ca_correct - 1) * 100:.1f} %")

CA correct : 13,591,643.70
CA gonflé  : 14,209,115.34
Écart      : 4.5 %


In [16]:
reviews = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv("/home/work/data/raw/olist_order_reviews_dataset.csv"))

print("Lignes :", reviews.count())
reviews.show(5, truncate=40)

Lignes : 99224
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------+--------------------+-----------------------+
|                       review_id|                        order_id|review_score|review_comment_title|                  review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------+--------------------+-----------------------+
|7bc2406110b926393aa56f80a40eba40|73fc7af87114b39712e6da79b0a377eb|           4|                NULL|                                    NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1ad469d5645fdfde|a548910a1c6147796b98fdf73dbeba33|           5|                NULL|                                    NULL| 2018-03-10 00:00:00|    2018-03-11 03:05:13|
|228ce5500dc1d8e020d8d1322874b6f0|f9e4b658b20

In [17]:
(reviews.groupBy("review_id").count()
        .filter("count > 1")
        .orderBy(F.desc("count"))
        .show(5, truncate=False))

+--------------------------------+-----+
|review_id                       |count|
+--------------------------------+-----+
|4d0e6dd087008d1f992d25ef6e1f619f|3    |
|0c76e7a547a531e7bf9f0b99cba071c1|3    |
|69a1068c3128a14994e3e422e4539e04|3    |
|e44840754f12fad2b8646712121b349a|3    |
|abbfacb2964f74f6487c9c10ac46daa6|3    |
+--------------------------------+-----+
only showing top 5 rows



In [18]:
reviews.filter(F.col("review_id") == "69a1068c3128a14994e3e422e4539e04").show(truncate=False)

+--------------------------------+--------------------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|69a1068c3128a14994e3e422e4539e04|175e7244409a1c34674cdd3407e25808|5           |NULL                |NULL                  |2017-09-06 00:00:00 |2017-09-08 00:24:52    |
|69a1068c3128a14994e3e422e4539e04|200331f78002771b79409a9cf7962dfa|5           |NULL                |NULL                  |2017-09-06 00:00:00 |2017-09-08 00:24:52    |
|69a1068c3128a14994e3e422e4539e04|30d7ab7638f1f3420e09474f704fe784|5           |NULL                |NULL                  |2017-09-06 00:00:00 |2017-

In [19]:
print("Lignes                      :", reviews.count())
print("review_id distincts         :", reviews.select("review_id").distinct().count())
print("Couples (review_id, order_id):", reviews.select("review_id", "order_id").distinct().count())
print("order_id distincts          :", reviews.select("order_id").distinct().count())

Lignes                      : 99224


review_id distincts         : 98410


Couples (review_id, order_id): 99224
order_id distincts          : 98673


In [21]:
df = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv("/home/work/data/raw/olist_geolocation_dataset.csv"))

print("Lignes :", geolocations.count())

NameError: name 'geolocations' is not defined

In [ ]:
df.printSchema()

In [ ]:

df.agg(
    countDistinct("geolocation_zip_code_prefix").alias("valeurs_distinctes"),
    count("*").alias("total_lignes")
).show()

In [ ]:
geo = tables["geolocation"]
clients = tables["customers"]

# Preuve du fan-out
print("Clients                       :", clients.count())
print("Après jointure directe        :",
      clients.join(geo, clients.customer_zip_code_prefix == geo.geolocation_zip_code_prefix).count())

In [ ]:
geo_agg = (geo
          .withColumn("lat", F.col("geolocation_lat").cast("double"))
          .withColumn("lng", F.col("geolocation_lng").cast("double"))
           .groupBy("geolocation_zip_code_prefix")
           .agg(F.avg("lat").alias("lat"), F.avg("lng").alias("lng")))
print("geo_agg lignes :", geo_agg.count())

In [ ]:
orders = tables["orders"]
items = tables["order_items"]

# Q5 : order_id dans order_items mais absents de orders
orphelins = items.join(orders, "order_id", "left_anti")
print("Lignes orphelines dans order_items :", orphelins.count())

# Bonus : commandes sans aucune ligne dans order_items (attendu : 775)
sans_produit = orders.join(items, "order_id", "left_anti")
print("Commandes sans produit :", sans_produit.count())

sans_produit.groupBy("order_status").count().orderBy(F.desc("count")).show()

In [ ]:
reviews = (spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv("/home/work/data/raw/olist_order_reviews_dataset.csv"))

In [ ]:
from pyspark.sql import Window, functions as F

w = Window.partitionBy("order_id").orderBy(F.col("review_answer_timestamp").desc())

reviews_par_commande = (reviews
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn"))

print("Lignes après dédoublonnage :", reviews_par_commande.count())
print("order_id distincts         :", reviews_par_commande.select("order_id").distinct().count())

In [22]:
for n, df in tables.items():
    print(f"| {n} | {df.count():,} | {len(df.columns)} | ... |".replace(",", " "))

| customers | 99 441 | 5 | ... |
| geolocation | 1 000 163 | 5 | ... |
| order_items | 112 650 | 7 | ... |
| order_payments | 103 886 | 5 | ... |
| order_reviews | 104 162 | 7 | ... |
| orders | 99 441 | 8 | ... |
| products | 32 951 | 9 | ... |
| sellers | 3 095 | 4 | ... |
| translation | 71 | 2 | ... |


26/11/08 20:45:46 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-b6b0b2ee-d41d-4188-b6c2-77a61478f9af/userFiles-7ac7cf16-d42d-4e7e-b8d1-26125c016bab. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-b6b0b2ee-d41d-4188-b6c2-77a61478f9af/userFiles-7ac7cf16-d42d-4e7e-b8d1-26125c016bab
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:173)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.SparkEnv.stop(SparkEnv.scala:108)
	at org.apache.spark.SparkContext.$anonfun$stop$25(SparkContext.scala:2310)
	at org.apac